In [ ]:
%pip install powerlaw networkx numpy scipy pandas tqdm matplotlib seaborn

In [ ]:
import sys
import logging
from pathlib import Path

# sys.path manipulation is required because sdt_netval is not installed as a package
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# scripts/ must be on sys.path for dynamic imports of numbered scripts (e.g., importlib.import_module("01_..."))
scripts_path = str(Path("../scripts").resolve())
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s")

from sdt_netval import load_network

db_path = Path("../data/00_raw/benchmark_runs/run01.sqlite").resolve()

G = load_network(db_path)

In [ ]:
import pandas as pd
from sdt_netval import GraphMetrics

report = GraphMetrics(G).generate_full_report()

pd.Series(report).rename("value").to_frame()

In [ ]:
from sdt_netval.pipeline import StageAValidator

data_dir = Path("../data/00_raw/benchmark_runs").resolve()

validator = StageAValidator(data_dir)
validator.process_runs()
validator.save_raw_results(Path("../data/01_processed/stage_a_raw.csv").resolve())

report = validator.full_stability_report()
print("\n=== Topological Stability (Stage A) ===\n")
print(report["stability"])

In [4]:
from importlib import import_module

stage_a_viz = import_module("02_stage_a_visualization")
StageAVisualizer = stage_a_viz.StageAVisualizer

viz = StageAVisualizer("../data/01_processed/stage_a_raw.csv")
viz.plot_stability_distributions("../data/01_processed/stage_a_stability.png")

# Optional: print text summary to stdout
print(viz.plot_summary_statistics())

02_stage_a_visualization — StageAVisualizer initialized — CSV loaded with 30 rows.
D:\University\Social Network Analysis\YSocial-Topology-Validator\scripts\02_stage_a_visualization.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
matplotlib.font_manager — Fontsize 0.00 < 1.0 pt not allowed by FreeType. Setting fontsize = 1 pt
D:\University\Social Network Analysis\YSocial-Topology-Validator\scripts\02_stage_a_visualization.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
matplotlib.font_manager — Fontsize 0.00 < 1.0 pt not allowed by FreeType. Setting fontsize = 1 pt
D:\University\Social Network Analysis\YSocial-Topology-Validator\scripts\02_stage_a_visualization.py:96: FutureWarn

=== Stability Report (Stage A) ===

Power-law Exponent (α)........ mean=2.3593, std=0.0489, CV=0.0207
Modularity (Q)................ mean=0.6082, std=0.0166, CV=0.0273
Average Clustering............ mean=0.0132, std=0.0032, CV=0.2442
Network Density............... mean=0.0003, std=0.0000, CV=0.0729



In [ ]:
from sdt_netval.pipeline import StageBAnalyzer

analyzer = StageBAnalyzer(Path("../data/00_raw/sensitivity_runs").resolve())
analyzer.process_all_runs()
analyzer.save_raw_results(Path("../data/01_processed/stage_b_raw.csv").resolve())
analyzer.save_aggregated_results(Path("../data/01_processed/stage_b_aggregated.csv").resolve())

report = analyzer.full_sensitivity_report()
print(report["aggregated"])

In [6]:
from importlib import import_module

stage_b_viz = import_module("04_stage_b_visualization")
StageBVisualizer = stage_b_viz.StageBVisualizer

viz = StageBVisualizer("../data/01_processed/stage_b_raw.csv")
viz.plot_modularity_comparison("../data/01_processed/stage_b_modularity_comparison.png")

# Optional: tabular modularity summary by condition
print(viz.get_summary_statistics())

04_stage_b_visualization — StageBVisualizer initialized — CSV loaded with 110 rows.
D:\University\Social Network Analysis\YSocial-Topology-Validator\scripts\04_stage_b_visualization.py:112: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
04_stage_b_visualization — Figure saved — file: 'D:\University\Social Network Analysis\YSocial-Topology-Validator\data\01_processed\stage_b_modularity_comparison.png'


  condition              label   n     mean       std       sem
0        c0           Baseline  10  0.64826  0.078156  0.024715
1        c1    Neutral Persona  10  0.49473  0.136959  0.043310
2        c3    Low Temperature  10  0.59936  0.085797  0.027131
3        c4   High Temperature  10  0.63588  0.041501  0.013124
4        c8  Aggressive RecSys  10  0.66275  0.069049  0.021835


In [7]:
from importlib import import_module

stage_b_hyp = import_module("05_stage_b_hypothesis")
StageBHypothesisTesting = stage_b_hyp.StageBHypothesisTesting

tester = StageBHypothesisTesting("../data/01_processed/stage_b_raw.csv")
results_df = tester.run_tests()

tester.print_results(verbose=True)

tester.save_results("../data/01_processed/stage_b_pvalues.csv")

# Conditions with p < 0.05
sig_conditions = tester.get_significant_conditions(alpha=0.05)
print(f"Significant conditions (p<0.05): {sig_conditions}")

05_stage_b_hypothesis — StageBHypothesisTesting initialized — 110 rows loaded.
05_stage_b_hypothesis — c0 vs c1: Δμ=-0.1535, U=86.00, p=0.007285 **
05_stage_b_hypothesis — c0 vs c3: Δμ=-0.0489, U=68.00, p=0.185877 ns
05_stage_b_hypothesis — c0 vs c4: Δμ=-0.0124, U=51.00, p=0.969839 ns
05_stage_b_hypothesis — c0 vs c8: Δμ=0.0145, U=40.00, p=0.472676 ns
05_stage_b_hypothesis — Test results saved — file: 'D:\University\Social Network Analysis\YSocial-Topology-Validator\data\01_processed\stage_b_pvalues.csv', rows: 4



MANN-WHITNEY U TEST RESULTS (Baseline: c0, Modularity)
Condition             Label  Mean_Baseline  Mean_Test  Mean_Difference  U_statistic  p_value Significance
       c1   Neutral Persona        0.64826    0.49473         -0.15353         86.0 0.007285           **
       c3   Low Temperature        0.64826    0.59936         -0.04890         68.0 0.185877           ns
       c4  High Temperature        0.64826    0.63588         -0.01238         51.0 0.969839           ns
       c8 Aggressive RecSys        0.64826    0.66275          0.01449         40.0 0.472676           ns
Significance: *** p<0.001 (highly significant), ** p<0.01 (very significant), * p<0.05 (significant), ns (not significant)

Significant conditions (p<0.05): ['c1']
